![header](img/header.PNG)
# Part 1 : Accessing and downloading GloFAS data from CEMS Early Warning Data Store

Welcome to the companion Jupyter notebook of the 4th CEMS Global Flood meeting -  Interactive session 2 (part 1): Accessing and downloading GloFAS data from CEMS Early Warning Data Store . This session explores the new Copernicus [Early Warning Data Store (EWDS)](https://ewds.climate.copernicus.eu).

This notebook will walk you through the basics of the new data store, inluding:
- How to create an account
- How to access data through the EWDS
- How to access data using our Python API clients
- How to use [earthkit](https://github.com/ecmwf/earthkit) to access and process the data.
- How to provide feedback and get support

## The Data Store

ECMWF operates the data store for the CEMS Early Warning Data Store:

<table border="1" style="width:100%; text-align:center;">
    <tr>
        <th>
            <h2 style="margin: 0; color: #FAA73E;">
                <a href="https://ewds.climate.copernicus.eu" target="_blank" 
                   style="text-decoration: none; color: #FAA73E;">Early Warning Data Store</a>
            </h2>
            <span style="color: #FAA73E;">Copernicus Emergency Management Service (CEMS)</span>
        </th>
    </tr>
</table>

The data stores provide free and open access to GloFAS and EFAS Data.

## Creating an account in the new Data Store

The following steps demonstrate how to register for the new Early Warning Data Store.

<div style="padding: 20px; background-color: #D4E5F7; border-left: 6px solid #006EAD; margin-bottom: 15px; width: 95%;">
    <strong>Important</strong>: If you previously had an account on the legacy Climate or Atmosphere Data Store, your old credentials will no longer work with the new Data Stores. Instead, you will need to associate your new EWDS account with an <strong>ECMWF</strong> account - but don't worry if you don't already have one, as you can create one as part of the registration process.
</div>

If this is the first time you have visited the EWDS, you will need to register for an account by clicking "Login - Register" in the top right-hand corner.

![Login Register](img/login-register.PNG)

This will take you to the ECMWF login/registration page, where you will need to either sign into your existing ECMWF account, or create a new one. For more information and support about registering for an ECMWF account, see the online documentation on [getting started with ECMWF services](https://confluence.ecmwf.int/display/UDOC/Getting+started+with+ECMWF+Services+and+Systems).

Once you are logged in to your ECMWF account, you will need to complete some basic information about yourself and your interests in the Climate Data Store, along with accepting the terms of use of the data store.

## Accessing data through the EWDS

The Early Warning Data Store provides a unified user interface for accessing data, offering a simple web form where you can select the specific subset of a dataset that you would like to download.

In this example, we will download data from [Glofas forecast](https://ewds.climate.copernicus.eu/datasets/cems-glofas-forecast?tab=overview) . To begin, navigate to the **[Datasets](https://ewds.climate.copernicus.eu/datasets)** page using the navigation bar at the top of the Early Warning Data Store, and search for *"glofas forecast"*. 
'River discharge and related forecasted data by Global Flood Awarness System ' will appear first in the list .

![Search Bar.PNG](img/ewds-search-bar.PNG)

Or you can use the filter in the top right corner to search for the dataset you're intreseted in .

![Filter.PNG](img/ewds-filter.PNG)

Once you have reached the *River discharge and related forecasted data by the Global Flood Awareness System* dataset, click on the title . This will take you to the download form, where you can manually select the data you are interested in.

As a simple example, let's explore today's forecast for a specific area of interest , let's say Australia and download different variables available .
Your selection should look something like this :
- System version : *Operational*
- Hydrological model : *LISFLOOD*
- Product Type : *Control forecast*
- Variable : *River discharge in the last 24 hours*
- Year : *2025*
- Month : *April*
- Day : *03*
- Leadtime hour : *24 to 720 (30 days)*

Since we're only interested in Australia, we can also use the geographical area selection tool to specify our domain of interest. Let's try going from -9°N to -39°S, and 109°W to 154°E.

![Area](img/area.PNG)

The last step is to select the Data format (Grib2/NetCDF-4) and Download format (Zip/Unarchived)

Once you have made your selection, accept the terms of use and click "Submit form" at the bottom of the page.

![Submit](img/submit-form.PNG)

This will take you to the "Your requests" page, which can also be viewed at any time by clicking *Your requests* in the top-right of the website.

The request you have just submitted will appear in your list of requests, which should look something like this:

![In progress](img/in-progress.PNG)

Once your request has completed, its status will change to **Complete** and you will be able to download your data by clicking the **Download** button.

![Download Completed](img/Download.PNG)

## Accessing data through the Python API

The Early Warning Data Store also provide programmatic access via the Python API. In this example, we will be accessing data from the EWDS with the [cdsapi Python package](https://github.com/ecmwf/cdsapi). If you get stuck while following this tutorial, you can follow more detailed instructions on this process at the [CDSAPI setup](https://ewds.climate.copernicus.eu/how-to-api) page in the EWDS.

While logged in to your EWDS account, you can visit your profile page by clicking your name in the top right of the EWDS, [or by following this link](https://ewds.climate.copernicus.eu/profile). Next, locate your **API Token** on this page, which should look something like this (please note that this is not your User ID):

![API Token ](img/API-Token-blur.png)

Next, you need to create a **`.cdsapirc`** file. By default, the cdsapi uses a file in your home directory (~/.cdsapirc) for credentials when connecting to the Data Stores.

The following code block will create/change this file for you. You should update `<API_TOKEN>` with your Personal Access Token from the previous step.

In [ ]:
## If you don't already have pyyaml installed, you'll need to uncomment 
## and run the pip install command below before running the next cell
# !pip install pyyaml

In [ ]:
import yaml
import os

HOME_DIR = os.path.expanduser("~")

# This is the URL for the CDS API
# If you would like to access a different data store, change this URL
URL = "https://ewds.climate.copernicus.eu/api"

# Replace <API TOKEN> with your own Personal Access Token
KEY = "<API_TOKEN>"

with open(f"{HOME_DIR}/.cdsapirc", "w") as stream:
    yaml.safe_dump(
        {"url": URL, "key": KEY},
        stream,
    )

## Make a request with the CDS API
Now that we have set up our `.cdsapirc`, we can make a request for EWDS data in a Python session. Let's make the same request we made earlier from the [River discharge and related forecasted data by Global Flood Awarness System](https://ewds.climate.copernicus.eu/datasets/cems-glofas-forecast?tab=download).

First of all, if you haven't already installed the cdsapi, uncomment (remove the #) and run the following cell:

In [ ]:
#!pip install cdsapi

Navigate to the [dataset page](https://ewds.climate.copernicus.eu/datasets/cems-glofas-forecast?tab=download) and fill in the form again - or alternatively, find the request you made earlier in the [Your requests](https://ewds.climate.copernicus.eu/requests?tab=all) page, then click *Details* followed by *Open request form*, which can be found next to the request ID.

Once you have completed the form, click the *Show API request code* box at the bottom of the page.
![Show-API-Request](img/Show-API-request.PNG)

This should give you the following Python code, which you can run to programmatically download the data from the EWDS:

In [ ]:
import cdsapi

dataset = "cems-glofas-forecast"
request = {
    "system_version": ["operational"],
    "hydrological_model": ["lisflood"],
    "product_type": ["control_forecast"],
    "variable": "river_discharge_in_the_last_24_hours",
    "year": ["2025"],
    "month": ["03"],
    "day": ["19"],
    "leadtime_hour": [
        "24",
        "48",
        "72",
        "96",
        "120",
        "144",
        "168",
        "192",
        "216",
        "240",
        "264",
        "288",
        "312",
        "336",
        "360",
        "384",
        "408",
        "432",
        "456",
        "480",
        "504",
        "528",
        "552",
        "576",
        "600",
        "624",
        "648",
        "672",
        "696",
        "720"
    ],
    "data_format": "grib2",
    "download_format": "zip",
    "area": [-9, 109, -39, 154]
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()


## Accessing Data Store Services data with earthkit

In the previous sections, we explored how to access data from the Copernicus Data Store Services through the **web** and through the **Python API**. In both of these cases, we downloaded the data directly onto our machine as GRIB files.

In the following section, we will use **[earthkit](https://github.com/ecmwf/earthkit)** to construct a *workflow* of data access, processing and visualisation.

**earthkit** is a Python package which provides tools for accessing, processing and visualisation earth science data. It also offers direct access to the Copernicus Data Store Services, speeding up the process of getting data from the Data Stores into your Python session.

If you do not already have **earthkit-data** and **earthkit-plots** installed, uncomment and run the following cell. We will be using **earthkit-data** to access data from the CDS, and **earthkit-plots** to visualise it.

In [ ]:
#!pip install earthkit-data
#!pip install earthkit-plots

Now we can use the `from_source()` method from **earthkit-data** to make the same request we made earlier to the CDS API.

**earthkit-data** uses the same interface as the CDS API, with one difference - the first argument to `earthkit.data.from_source()` needs to be `"cds"` if we're accessing CDS data. After that, just like the `retrieve` method from the CDS API, we need to pass the name of the dataset followed by the request details.

In [ ]:
import earthkit.data

dataset = "cems-glofas-forecast"
request = {
    "system_version": ["operational"],
    "hydrological_model": ["lisflood"],
    "product_type": ["control_forecast"],
    "variable": "river_discharge_in_the_last_24_hours",
    "year": ["2025"],
    "month": ["03"],
    "day": ["19"],
    "leadtime_hour": [
        "24",
        "48",
        "72",
        "96",
        "120",
        "144",
        "168",
        "192",
        "216",
        "240",
        "264",
        "288",
        "312",
        "336",
        "360",
        "384",
        "408",
        "432",
        "456",
        "480",
        "504",
        "528",
        "552",
        "576",
        "600",
        "624",
        "648",
        "672",
        "696",
        "720"
    ],
    "data_format": "grib2",
    "download_format": "zip",
    "area": [-9, 109, -39, 154]
}


data = earthkit.data.from_source("cds", dataset, request)

One advantage of using **earthkit** instead of the CDS API is that the object returned from our request can be immediately opened and explored with commonly used scientific tools like `xarray`:

In [ ]:
data.to_xarray()

<div style="padding: 20px; background-color: #D4E5F7; border-left: 6px solid #006EAD; margin-bottom: 15px; width: 95%;"><strong>
For help concerning any problem you encounter while dealing with EWDS and CEMS data , please raise a request here</strong><a href=https://ewds.climate.copernicus.eu/help> Help and support</a></div>